### Get JPEG world file affine parameters, if they exist

In [ ]:
gdf = gpd.read_file("data/historic_airphoto_index/index_table_cleaned_pre-json.gpkg")

In [ ]:
def fetch_worldfile(jpg_url):
    """Retrieve a JGW and return its affine parameters as JSON."""

    if not isinstance(jpg_url, str) or not jpg_url:
        return None

    # Replace .jpg with .jgw
    jgw_url = jpg_url.removesuffix(".jpg") + ".jgw"

    try:
        response = requests.get(jgw_url, timeout=15)

        # Confirmed missing world file
        if response.status_code in (300, 404):
            return "no jgw"

        response.raise_for_status()

        # Some missing files may return an HTML page
        if "text/html" in response.headers.get("Content-Type", "").lower():
            return "no jgw" if "Multiple Choices" in response.text else None

        # World-file parameter order: A, D, B, E, C, F
        values = [float(v) for v in response.text.split()]

        if len(values) != 6:
            raise ValueError("World file must contain exactly 6 values")

        parameters = dict(zip(
            ["A", "D", "B", "E", "C", "F"],
            values
        ))

        return json.dumps(parameters)

    except (requests.RequestException, ValueError) as e:
        print(f"Failed to retrieve {jgw_url}: {e}")
        return None

In [ ]:
test_json = fetch_worldfile("https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/stl/y1989/stl_028_107.jpg")

print(test_json)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from pathlib import Path
import pandas as pd

output_path = Path(
    "data/historic_airphoto_index/index_with_jgw_jsons.gpkg"
)

# Resume from previous saved progress, if available
if output_path.exists():
    gdf = gpd.read_file(output_path)
    print("Loaded previously saved progress.")

# Find records that haven't been checked
pending = gdf.loc[gdf["worldfile_json"].isna(), "url"]

print(f"Remaining URLs to check: {len(pending):,}")

save_every = 500
completed = 0

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(fetch_worldfile, url): idx
        for idx, url in pending.items()
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Fetching world files"
    ):
        idx = futures[future]

        # Store the result immediately
        gdf.at[idx, "worldfile_json"] = future.result()

        completed += 1

        # Save progress every 500 requests
        if completed % save_every == 0:
            gdf.to_file(output_path, driver="GPKG")

# Final save
gdf.to_file(output_path, driver="GPKG")

print(gdf["worldfile_json"].notna().value_counts())